# 网络爬虫的基本框架

运行准备：复用示例代码 3.23 中的 `PageDownloader` 类和测试网址。

In [ ]:
import time, random, requests
from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup
from p3lib.ch3 import PageDownloader

test_url = "http://www.moe.gov.cn/srcsite/A22/s7065/200612/t20061206_128833.html"


In [ ]:
class WebCrawler(PageDownloader):
    def __init__(self, trace_link=True, save_dir="html_pages"):
        super().__init__(save_dir)
        self.trace_link = trace_link
        self.task_queue = []
        self.url_cache = set() # 已访问 URL 集合，避免重复爬取

    def add(self, url):
        if url not in self.url_cache:
            self.task_queue.append(url)

    def addmany(self, urls):
        for url in urls:
            self.add(url)

    def fetch(self):
        print(f"[Status] 当前工作队列长度：{len(self.task_queue)}")
        if self.task_queue:
            return self.task_queue.pop(0)
        else:
            return None

    def parse(self, htmlstr, base_url):
        soup = BeautifulSoup(htmlstr, 'html.parser')
        new_links = set()
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href'].strip()
            if not href or href.startswith(('javascript:', 'mailto:', 'tel:')): # 跳过无效链接
                continue
            link = urljoin(base_url, href) # 规范化 URL（转换为绝对路径并移除片段标识符）
            parsed = urlparse(link)
            if parsed.fragment:
                link = parsed._replace(fragment='').geturl()
            new_links.add(link)
        return list(new_links)

    def process(self, url):
        response = super().process(url)
        self.url_cache.add(url)
        if response and self.trace_link:
            new_seeds = self.parse(response.text, url)
            self.addmany(new_seeds)
        return response

    def run(self):
        url = self.fetch()
        while url:
            res = self.process(url)
            if len(self.url_cache) >= 100: # 抓取了 100 个网页后就强制结束
                break
            sleep_time = random.uniform(1, 5)
            print(f"[Status] Sleeping for {sleep_time:.2f} seconds...")
            time.sleep(sleep_time) # 随机休眠，以免影响网站正常运行
            url = self.fetch()
        print(f"[Status] {self.__class__.__name__} stopped after {len(self.url_cache)} pages visited.")


In [ ]:
"""测试代码"""
wc = WebCrawler()
wc.add(test_url)
wc.run()
